In [ ]:
import os
from deepfix_sdk import DeepFixClient

In [ ]:
os.environ["DEEPFIX_API_KEY"] = "sk-empty"

In [14]:
client = DeepFixClient(api_url="https://deepfix.delcaux.com", timeout=120)

## Classification

In [15]:
from deepfix_sdk.data.datasets import TabularDataset
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

In [ ]:
# Load data
X, y = load_breast_cancer(as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
dataset_name = "breast_cancer_classification"

label = "target"
train = X_train.copy()
train[label] = y_train
cat_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
if len(cat_features) > 0:
    cat_features = None

test = X_test.copy()
test[label] = y_test

train_data = TabularDataset(
    dataset=train, dataset_name=dataset_name, label=label, cat_features=cat_features
)
val_data = TabularDataset(
    dataset=test, dataset_name=dataset_name, label=label, cat_features=cat_features
)

In [17]:
train_data.get_data().head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
546,10.32,16.35,65.31,324.9,0.09434,0.04994,0.01012,0.005495,0.1885,0.06201,...,21.77,71.12,384.9,0.1285,0.08842,0.04384,0.02381,0.2681,0.07399,1
432,20.18,19.54,133.80,1250.0,0.11330,0.14890,0.21330,0.125900,0.1724,0.06053,...,25.07,146.00,1479.0,0.1665,0.29420,0.53080,0.21730,0.3032,0.08075,0
174,10.66,15.15,67.49,349.6,0.08792,0.04302,0.00000,0.000000,0.1928,0.05975,...,19.20,73.20,408.3,0.1076,0.06791,0.00000,0.00000,0.2710,0.06164,1
221,13.56,13.90,88.59,561.3,0.10510,0.11920,0.07860,0.044510,0.1962,0.06303,...,17.13,101.10,686.6,0.1376,0.26980,0.25770,0.09090,0.3065,0.08177,1
289,11.37,18.89,72.17,396.0,0.08713,0.05008,0.02399,0.021730,0.2013,0.05955,...,26.14,79.29,459.3,0.1118,0.09708,0.07529,0.06203,0.3267,0.06994,1


In [ ]:
# Fit model
model_name = "HistGradientBoostingClassifier"
clf = HistGradientBoostingClassifier(max_depth=3)
clf = clf.fit(train_data.X, train_data.y)

In [ ]:
result = client.get_diagnosis(
    train_data=train_data,
    test_data=val_data,
    model_name=model_name,
    model=clf,
    language="english",
)

In [20]:
# Visualize results
result.to_text(verbose=False)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ The cross-artifact analysis reveals a critical gap between theoretical model performance and deployment readiness.   │
│ While the Deepchecks analysis shows excellent model performance (AUC ~1.0) with some feature optimization            │
│ opportunities (high redundancy, unused features), the ModelCheckpoint analysis exposes that the model cannot be      │
│ deployed due to missing trained model files and essential metadata. The highest priority is generating the actual    │
│ model artifact. Once available, feature optimization should be addressed to improve model stability and leverage     │
│ unused high-variance features. The excellent performance metrics are promising but meaningless without executable    │
│ model files.                                                                                                         │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  4                                                            
 Severity Distribution           MEDIUM: 2  HIGH: 1  LOW: 1                                   

                                  HIGH Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Critical deployment blocker: Missing     │ Generate and include the actual trained  │
│     │ trained model file                       │ model file (pickle, joblib, or ONNX      │
│     │ Evidence: ModelCheckpoint analyzer       │ format) with the configuration artifacts │
│     │ confirms no actual model weights or      │ The model configuration alone is         │
│     │ serialized model object exists, only     │ insufficient for deployment or           │
│     │ configuration parameters. Deepchecks     │ inference. The excellent performance     │
│     │ analysis shows excellent theoretical     │ metrics from Deepchecks are meaningless  │
│     │ performance (AUC ~1.0) but this cannot   │ without the executable model.            │
│     │ be utilized without the model file.      │                                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (2)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Incomplete model metadata hinders        │ Include feature_names_in_, classes_, and │
│     │ deployment                               │ n_features_in_ attributes with the       │
│     │ Evidence: Missing feature names, class   │ trained model, and review feature        │
│     │ labels, and feature count metadata in    │ selection strategy                       │
│     │ ModelCheckpoint, while Deepchecks shows  │ Proper metadata is essential for         │
│     │ 16 high-variance features are unused and │ deployment. The feature issues           │
│     │ 22 feature pairs have high correlation   │ (redundancy, unused features) from       │
│     │ (>0.9)                                   │ Deepchecks analysis should inform which  │
│     │                                          │ features to include in the final         │
│     │                                          │ deployed model.                          │
│ 2   │ Feature optimization opportunity despite │ Perform feature selection (PCA or        │
│     │ excellent performance                    │ representative feature selection) and    │
│     │ Evidence: Deepchecks shows high feature  │ investigate incorporating unused         │
│     │ redundancy (22 correlated pairs >0.9)    │ high-variance features                   │
│     │ and 16 unused high-variance features,    │ Reducing multicollinearity can improve   │
│     │ while model maintains AUC ~1.0 with F1   │ model stability and interpretability.    │
│     │ gain of 91.49%                           │ The unused features may contain valuable │
│     │                                          │ signal for robustness.                   │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                   LOW Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Excellent model performance with         │ Monitor performance on new data and      │
│     │ potential overfitting risk               │ conduct feature importance analysis to   │
│     │ Evidence: Deepchecks reports             │ validate robustness                      │
│     │ near-perfect performance (AUC ~1.0) but  │ While current performance is             │
│     │ with heavy reliance on 3 features        │ exceptional, reliance on few features    │
│     │ showing very high predictive power (PPS  │ may make the model vulnerable to         │
│     │ > 0.7)                                   │ distribution shifts in production.       │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''